# 01 - Introducao ao ARIMA(p,d,q) - SOLUTION

> **Este notebook contem a solucao completa de todos os exercicios.**
> Todas as celulas do tutorial original estao incluidas, alem das resolucoes dos exercicios propostos.

## 1. Fundamentacao Teorica

O modelo **ARIMA(p, d, q)** combina tres componentes:

| Componente | Sigla | Descricao |
|---|---|---|
| **AR(p)** | Auto-Regressivo | Usa `p` valores passados da serie como preditores |
| **I(d)** | Integrado | Aplica `d` diferenciacoes para tornar a serie estacionaria |
| **MA(q)** | Medias Moveis | Usa `q` erros passados como preditores |

### Forma geral

$$\phi(B)(1-B)^d Y_t = \theta(B) \varepsilon_t$$

Onde:
- $\phi(B) = 1 - \phi_1 B - \phi_2 B^2 - \ldots - \phi_p B^p$ (polinomio AR)
- $\theta(B) = 1 + \theta_1 B + \theta_2 B^2 + \ldots + \theta_q B^q$ (polinomio MA)
- $B$ e o operador de retardo ($B Y_t = Y_{t-1}$)
- $\varepsilon_t \sim WN(0, \sigma^2)$ (ruido branco)

### Metodologia de Box-Jenkins

1. **Identificacao**: Determinar p, d, q usando ACF, PACF e testes de estacionariedade
2. **Estimacao**: Estimar os parametros do modelo via Maxima Verossimilhanca
3. **Diagnostico**: Verificar se os residuos sao ruido branco
4. **Previsao**: Gerar previsoes com intervalos de confianca

## 2. Setup e Carregamento dos Dados

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import os

from chronobox import ARIMA
from chronobox.tests_stat import adf_test, kpss_test, ljung_box_test, jarque_bera_test
from chronobox.visualization import plot_diagnostics, plot_forecast, plot_series, set_theme

set_theme('professional')
np.random.seed(42)

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')
OUTPUT_DIR = os.path.join(os.path.dirname(os.getcwd()), 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('chronobox importado com sucesso!')

chronobox importado com sucesso!


In [2]:
nile = pd.read_csv(os.path.join(DATA_DIR, 'nile.csv'), parse_dates=['date'])
nile.set_index('date', inplace=True)
y = nile['flow'].values

print(f'Periodo: {nile.index[0].year} - {nile.index[-1].year}')
print(f'Observacoes: {len(y)}')
print(f'Media: {y.mean():.1f}, Desvio: {y.std():.1f}')
nile.head()

Periodo: 1871 - 1970
Observacoes: 100
Media: 919.4, Desvio: 168.4


,flow
date,
1871-01-01,1120
1872-01-01,1160
1873-01-01,963
1874-01-01,1210
1875-01-01,1160


## 3. Analise Exploratoria e Estacionariedade

In [3]:
fig = plot_series(y, labels=['Vazao do Nilo'], title='Vazao Anual do Rio Nilo (1871-1970)')
plt.xlabel('Observacao')
plt.ylabel('Vazao (10^8 m^3)')
plt.show()

/tmp/ipykernel_95438/2807699831.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# Teste ADF (H0: serie possui raiz unitaria = nao estacionaria)
adf_result = adf_test(y, regression='c')
print('=== Teste ADF ===')
print(f'Estatistica: {adf_result.statistic:.4f}')
print(f'p-valor: {adf_result.pvalue:.4f}')
print(f'Conclusao: {"Rejeita H0 -> Estacionaria" if adf_result.pvalue < 0.05 else "Nao rejeita H0 -> Nao estacionaria"}')

print()

# Teste KPSS (H0: serie e estacionaria)
kpss_result = kpss_test(y, regression='c')
print('=== Teste KPSS ===')
print(f'Estatistica: {kpss_result.statistic:.4f}')
print(f'p-valor: {kpss_result.pvalue:.4f}')
print(f'Conclusao: {"Rejeita H0 -> Nao estacionaria" if kpss_result.pvalue < 0.05 else "Nao rejeita H0 -> Estacionaria"}')

=== Teste ADF ===
Estatistica: -4.0487
p-valor: 0.0050
Conclusao: Rejeita H0 -> Estacionaria

=== Teste KPSS ===
Estatistica: 0.9654
p-valor: 0.0050
Conclusao: Rejeita H0 -> Nao estacionaria


In [5]:
# Primeira diferenca
y_diff = np.diff(y)

adf_diff = adf_test(y_diff, regression='c')
print('=== ADF na serie diferenciada ===')
print(f'Estatistica: {adf_diff.statistic:.4f}')
print(f'p-valor: {adf_diff.pvalue:.4f}')
print(f'Conclusao: d=1 torna a serie {"estacionaria" if adf_diff.pvalue < 0.05 else "nao estacionaria"}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(y, color='steelblue')
axes[0].set_title('Serie Original')
axes[1].plot(y_diff, color='darkorange')
axes[1].set_title('Primeira Diferenca (d=1)')
plt.tight_layout()
plt.show()

=== ADF na serie diferenciada ===
Estatistica: -4.6797
p-valor: 0.0050
Conclusao: d=1 torna a serie estacionaria


/tmp/ipykernel_95438/3253117256.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Identificacao da Ordem (p, q) via ACF e PACF

| Modelo | ACF | PACF |
|---|---|---|
| AR(p) | Decaimento exponencial ou senoidal | Corte abrupto apos lag p |
| MA(q) | Corte abrupto apos lag q | Decaimento exponencial ou senoidal |
| ARMA(p,q) | Decaimento apos lag q | Decaimento apos lag p |

In [6]:
from chronobox.visualization.diagnostics_plot import _compute_acf, _compute_pacf

max_lag = 20
acf_vals = _compute_acf(y_diff, max_lag)
pacf_vals = _compute_pacf(y_diff, max_lag)

ci = 1.96 / np.sqrt(len(y_diff))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(max_lag + 1), acf_vals, width=0.3, color='steelblue')
axes[0].axhline(ci, color='red', linestyle='--', alpha=0.7)
axes[0].axhline(-ci, color='red', linestyle='--', alpha=0.7)
axes[0].set_title('ACF - Serie Diferenciada (d=1)')
axes[0].set_xlabel('Lag')

axes[1].bar(range(max_lag + 1), pacf_vals, width=0.3, color='darkorange')
axes[1].axhline(ci, color='red', linestyle='--', alpha=0.7)
axes[1].axhline(-ci, color='red', linestyle='--', alpha=0.7)
axes[1].set_title('PACF - Serie Diferenciada (d=1)')
axes[1].set_xlabel('Lag')

plt.tight_layout()
plt.show()

print('ACF: possivel corte apos lag 1 -> q=1')
print('PACF: possivel corte apos lag 1 -> p=1')
print('Sugestao inicial: ARIMA(1,1,1)')

ACF: possivel corte apos lag 1 -> q=1
PACF: possivel corte apos lag 1 -> p=1
Sugestao inicial: ARIMA(1,1,1)


/tmp/ipykernel_95438/820779888.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Ajuste do Modelo ARIMA com chronobox

In [7]:
model = ARIMA(order=(1, 1, 1))
results = model.fit(y)
print(results.summary())

                         Model: ARIMA(1,1,1)                          
  Nobs: 100    Effective Nobs: 99
  Log-Likelihood: -630.6274
  AIC: 1267.2548    BIC: 1275.0401    AICc: 1267.5074    HQIC: 1270.4047
----------------------------------------------------------------------
  Parameter         Estimate    Std.Err    t-value    p-value
----------------------------------------------------------------------
  ar.L1               0.2544     0.1194     2.1299     0.0357
  ma.L1              -0.8741     0.0605   -14.4538     0.0000
  sigma2          19769.3088    34.2460   577.2742     0.0000
  Residual std: 140.3736    Mean: -16.6251
  Ljung-Box(lag=10): stat=9.5634  p-value=0.4796


In [8]:
# Comparacao de modelos
modelos = {
    'ARIMA(1,1,0)': ARIMA(order=(1, 1, 0)),
    'ARIMA(0,1,1)': ARIMA(order=(0, 1, 1)),
    'ARIMA(1,1,1)': ARIMA(order=(1, 1, 1)),
    'ARIMA(2,1,1)': ARIMA(order=(2, 1, 1)),
}

resultados = {}
print(f'{"Modelo":<18} {"AIC":>10} {"BIC":>10} {"Log-Lik":>10}')
print('-' * 50)

for nome, mod in modelos.items():
    res = mod.fit(y)
    resultados[nome] = res
    print(f'{nome:<18} {res.aic:>10.2f} {res.bic:>10.2f} {res.loglike:>10.2f}')

melhor = min(resultados, key=lambda k: resultados[k].aic)
print(f'\nMelhor modelo por AIC: {melhor}')

Modelo                    AIC        BIC    Log-Lik
--------------------------------------------------
ARIMA(1,1,0)          1281.48    1286.67    -638.74
ARIMA(0,1,1)          1269.09    1274.28    -632.55
ARIMA(1,1,1)          1267.25    1275.04    -630.63


ARIMA(2,1,1)          1268.90    1279.28    -630.45

Melhor modelo por AIC: ARIMA(1,1,1)


## 6. Diagnostico de Residuos

In [9]:
best_result = resultados[melhor]
fig = plot_diagnostics(best_result, lags=20, title=f'Diagnostico - {melhor} (Nile)')
plt.show()

/tmp/ipykernel_95438/1277284443.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
resid = best_result.residuals

# Ljung-Box
for lag in [10, 15, 20]:
    lb = ljung_box_test(resid, lags=lag, model_df=2)
    print(f'Ljung-Box(lag={lag}): Q={lb.statistic:.3f}, p={lb.pvalue:.4f}')

print()

# Jarque-Bera
jb = jarque_bera_test(resid)
print(f'Jarque-Bera: stat={jb.statistic:.3f}, p={jb.pvalue:.4f}')
print(f'Residuos normais: {"Sim" if jb.pvalue > 0.05 else "Nao"}')

Ljung-Box(lag=10): Q=9.563, p=0.2970
Ljung-Box(lag=15): Q=11.293, p=0.5863
Ljung-Box(lag=20): Q=12.653, p=0.8118

Jarque-Bera: stat=0.199, p=0.9053
Residuos normais: Sim


## 7. Previsao com Intervalos de Confianca

In [11]:
fc = best_result.forecast(steps=10, alpha=0.05)

df_fc = pd.DataFrame({
    'Step': range(1, 11),
    'Previsao': fc['forecast'],
    'IC Inferior (95%)': fc['lower'],
    'IC Superior (95%)': fc['upper']
})
df_fc['Amplitude IC'] = df_fc['IC Superior (95%)'] - df_fc['IC Inferior (95%)']
print(df_fc.to_string(index=False, float_format='%.2f'))

 Step  Previsao  IC Inferior (95%)  IC Superior (95%)  Amplitude IC
    1     76.18            -199.40             351.76        551.16
    2     19.38            -304.83             343.59        648.42
    3      4.93            -322.18             332.04        654.22
    4      1.25            -326.04             328.55        654.59
    5      0.32            -326.99             327.63        654.62
    6      0.08            -327.23             327.39        654.62
    7      0.02            -327.29             327.33        654.62
    8      0.01            -327.30             327.31        654.62
    9      0.00            -327.31             327.31        654.62
   10      0.00            -327.31             327.31        654.62


In [12]:
# Grafico de previsao
n_show = 40
fc = best_result.forecast(steps=10, alpha=0.05)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(len(y)-n_show, len(y)), y[-n_show:], color='steelblue', linewidth=1.5, label='Observado')
x_fc = range(len(y), len(y)+10)
ax.plot(x_fc, fc['forecast'], color='red', linewidth=2, label='Previsao')
ax.fill_between(x_fc, fc['lower'], fc['upper'], alpha=0.2, color='red', label='IC 95%')
ax.axvline(len(y)-1, color='gray', linestyle=':', alpha=0.5)
ax.set_title(f'Previsao {melhor} - Vazao do Nilo')
ax.set_xlabel('Observacao')
ax.set_ylabel('Vazao (10^8 m^3)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

/tmp/ipykernel_95438/1540328702.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
# Valores ajustados vs observados
fitted = best_result.fitted_values
n_fitted = len(fitted)
y_plot = y[-n_fitted:]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(len(y)-n_fitted, len(y)), y_plot, color='steelblue', linewidth=1, label='Observado', alpha=0.8)
ax.plot(range(len(y)-n_fitted, len(y)), fitted, color='red', linewidth=1, label='Ajustado', alpha=0.8)
ax.set_title(f'Valores Ajustados vs Observados - {melhor}')
ax.set_xlabel('Observacao')
ax.set_ylabel('Vazao')
ax.legend()
plt.tight_layout()
plt.show()

# Metricas de ajuste
valid = ~np.isnan(fitted)
mae = np.mean(np.abs(y_plot[valid] - fitted[valid]))
rmse = np.sqrt(np.mean((y_plot[valid] - fitted[valid])**2))
print(f'MAE: {mae:.2f}')
print(f'RMSE: {rmse:.2f}')

MAE: 904.54
RMSE: 927.59


/tmp/ipykernel_95438/4117550294.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Exercicio 1: Ajustar ARIMA no dataset airline.csv - SOLUCAO

In [14]:
# Exercicio 1 - SOLUCAO COMPLETA
# 1. Carregar dataset airline.csv
airline = pd.read_csv(os.path.join(DATA_DIR, 'airline.csv'), parse_dates=['date'])
airline.set_index('date', inplace=True)
y_air = airline['passengers'].values

print(f'Periodo: {airline.index[0]} a {airline.index[-1]}')
print(f'Observacoes: {len(y_air)}')
print(f'Media: {y_air.mean():.1f}, Desvio: {y_air.std():.1f}')

# 2. Plotar serie temporal
fig = plot_series(y_air, labels=['Passageiros'], title='Passageiros Aereos (1949-1960)')
plt.xlabel('Mes')
plt.ylabel('Milhares de passageiros')
plt.show()

print('Observa-se tendencia crescente e sazonalidade multiplicativa (amplitude cresce).')
print('Para ARIMA puro, usaremos log para estabilizar variancia.')

Periodo: 1949-01-01 00:00:00 a 1960-12-01 00:00:00
Observacoes: 144
Media: 280.3, Desvio: 119.5
Observa-se tendencia crescente e sazonalidade multiplicativa (amplitude cresce).
Para ARIMA puro, usaremos log para estabilizar variancia.


/tmp/ipykernel_95438/3671821391.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# 3. Transformacao log e diferenciacoes
y_log = np.log(y_air)

# ADF na serie original (log)
adf_orig = adf_test(y_log, regression='c')
print(f'ADF em log(airline): stat={adf_orig.statistic:.4f}, p={adf_orig.pvalue:.4f}')

# Primeira diferenca
y_log_diff = np.diff(y_log)
adf_diff = adf_test(y_log_diff, regression='c')
print(f'ADF em diff(log(airline)): stat={adf_diff.statistic:.4f}, p={adf_diff.pvalue:.4f}')
print(f'Conclusao: d=1 torna a serie {"estacionaria" if adf_diff.pvalue < 0.05 else "nao estacionaria"}')

ADF em log(airline): stat=-1.7170, p=0.4007
ADF em diff(log(airline)): stat=-3.0530, p=0.0387
Conclusao: d=1 torna a serie estacionaria


In [16]:
# 4. ACF e PACF da serie diferenciada
from chronobox.visualization.diagnostics_plot import _compute_acf, _compute_pacf

max_lag = 24
acf_vals = _compute_acf(y_log_diff, max_lag)
pacf_vals = _compute_pacf(y_log_diff, max_lag)
ci = 1.96 / np.sqrt(len(y_log_diff))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(range(max_lag + 1), acf_vals, width=0.3, color='steelblue')
axes[0].axhline(ci, color='red', linestyle='--', alpha=0.7)
axes[0].axhline(-ci, color='red', linestyle='--', alpha=0.7)
axes[0].set_title('ACF - log(airline) diferenciada')

axes[1].bar(range(max_lag + 1), pacf_vals, width=0.3, color='darkorange')
axes[1].axhline(ci, color='red', linestyle='--', alpha=0.7)
axes[1].axhline(-ci, color='red', linestyle='--', alpha=0.7)
axes[1].set_title('PACF - log(airline) diferenciada')

plt.tight_layout()
plt.show()

print('Nota: Picos nos lags 12, 24 indicam sazonalidade nao capturada pelo ARIMA puro.')
print('ACF com decaimento lento + picos sazonais -> AR(1) ou AR(2) como aproximacao.')

Nota: Picos nos lags 12, 24 indicam sazonalidade nao capturada pelo ARIMA puro.
ACF com decaimento lento + picos sazonais -> AR(1) ou AR(2) como aproximacao.


/tmp/ipykernel_95438/1137692589.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
# 5. Ajustar ARIMA(1,1,1) e ARIMA(2,1,2) no airline (conforme spec)
modelos_air = {
    'ARIMA(1,1,0)': ARIMA(order=(1, 1, 0)),
    'ARIMA(0,1,1)': ARIMA(order=(0, 1, 1)),
    'ARIMA(1,1,1)': ARIMA(order=(1, 1, 1)),
    'ARIMA(2,1,1)': ARIMA(order=(2, 1, 1)),
    'ARIMA(2,1,2)': ARIMA(order=(2, 1, 2)),
}

print(f'{"Modelo":<18} {"AIC":>10} {"BIC":>10} {"Log-Lik":>10}')
print('-' * 50)

resultados_air = {}
for nome, mod in modelos_air.items():
    res = mod.fit(y_log)
    resultados_air[nome] = res
    print(f'{nome:<18} {res.aic:>10.2f} {res.bic:>10.2f} {res.loglike:>10.2f}')

melhor_air = min(resultados_air, key=lambda k: resultados_air[k].aic)
print(f'\nMelhor modelo por AIC: {melhor_air}')
print('Nota: ARIMA puro nao captura sazonalidade - SARIMA seria mais adequado.')

Modelo                    AIC        BIC    Log-Lik
--------------------------------------------------
ARIMA(1,1,0)          -236.60    -230.67     120.30


ARIMA(0,1,1)          -238.73    -232.80     121.36

ARIMA(1,1,1)          -238.60    -229.72     122.30


/home/guhaase/.local/lib/python3.12/site-packages/scipy/optimize/_numdiff.py:686: RuntimeWarning: invalid value encountered in subtract
  df = [f_eval - f0 for f_eval in f_evals]


ARIMA(2,1,1)          -247.07    -235.22     127.54


ARIMA(2,1,2)          -246.87    -232.06     128.44

Melhor modelo por AIC: ARIMA(2,1,1)
Nota: ARIMA puro nao captura sazonalidade - SARIMA seria mais adequado.


In [18]:
# Diagnostico do ARIMA(1,1,1) no airline
res_air_111 = resultados_air['ARIMA(1,1,1)']
fig = plot_diagnostics(res_air_111, lags=24, title='Diagnostico - ARIMA(1,1,1) Airline')
plt.show()

# Ljung-Box
resid_air = res_air_111.residuals
for lag in [12, 24]:
    lb = ljung_box_test(resid_air, lags=lag, model_df=2)
    print(f'Ljung-Box(lag={lag}): Q={lb.statistic:.3f}, p={lb.pvalue:.4f}')

print('\nResiduos mostram autocorrelacao sazonal - confirma necessidade de SARIMA.')

Ljung-Box(lag=12): Q=130.581, p=0.0000
Ljung-Box(lag=24): Q=249.200, p=0.0000

Residuos mostram autocorrelacao sazonal - confirma necessidade de SARIMA.


/tmp/ipykernel_95438/799331186.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Exercicio 2: Impacto da Ordem de Diferenciacao - SOLUCAO

In [19]:
# Exercicio 2 - SOLUCAO COMPLETA
# Ajustar ARIMA(1,d,1) com d = 0, 1, 2 no Nile
modelos_d = {
    'ARIMA(1,0,1)': ARIMA(order=(1, 0, 1)),
    'ARIMA(1,1,1)': ARIMA(order=(1, 1, 1)),
    'ARIMA(1,2,1)': ARIMA(order=(1, 2, 1)),
}

resultados_d = {}
print(f'{"Modelo":<18} {"AIC":>10} {"BIC":>10}')
print('-' * 40)
for nome, mod in modelos_d.items():
    res = mod.fit(y)
    resultados_d[nome] = res
    print(f'{nome:<18} {res.aic:>10.2f} {res.bic:>10.2f}')

Modelo                    AIC        BIC
----------------------------------------
ARIMA(1,0,1)          1282.13    1292.55
ARIMA(1,1,1)          1267.25    1275.04
ARIMA(1,2,1)          1276.70    1284.45


In [20]:
# Previsoes lado a lado
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
steps = 15
n = len(y)

for ax, (nome, res) in zip(axes, resultados_d.items()):
    fc = res.forecast(steps=steps)
    ax.plot(range(n-30, n), y[-30:], color='steelblue', linewidth=1.5, label='Observado')
    ax.plot(range(n, n+steps), fc['forecast'], color='red', linewidth=2, label='Previsao')
    ax.fill_between(range(n, n+steps), fc['lower'], fc['upper'], alpha=0.2, color='red')
    ax.set_title(nome)
    ax.legend(fontsize=8)
    ax.set_xlabel('Observacao')

plt.suptitle('Impacto da Ordem de Diferenciacao nas Previsoes', fontsize=13)
plt.tight_layout()
plt.show()

print('Observacoes:')
print('- d=0: previsao reverte para a media; IC estreito')
print('- d=1: previsao plana (random walk); IC cresce linearmente')
print('- d=2: previsao segue tendencia local; IC cresce rapidamente (sobre-diferenciacao)')
print('- Quanto maior d, maior a variancia da previsao de longo prazo')

Observacoes:
- d=0: previsao reverte para a media; IC estreito
- d=1: previsao plana (random walk); IC cresce linearmente
- d=2: previsao segue tendencia local; IC cresce rapidamente (sobre-diferenciacao)
- Quanto maior d, maior a variancia da previsao de longo prazo


/tmp/ipykernel_95438/2969528562.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Salvando Resultados para Comparacao

In [21]:
# Salvar coeficientes em outputs/arima_coefficients.json
all_coefficients = {}

# Modelos no Nile
for nome, res in resultados.items():
    coefs = {}
    for pname, pval in zip(res.param_names, res.params):
        coefs[pname] = round(float(pval), 6)
    coefs['aic'] = round(float(res.aic), 4)
    coefs['bic'] = round(float(res.bic), 4)
    coefs['loglike'] = round(float(res.loglike), 4)
    all_coefficients[f'nile_{nome}'] = coefs

# Modelos no airline
for nome, res in resultados_air.items():
    coefs = {}
    for pname, pval in zip(res.param_names, res.params):
        coefs[pname] = round(float(pval), 6)
    coefs['aic'] = round(float(res.aic), 4)
    coefs['bic'] = round(float(res.bic), 4)
    coefs['loglike'] = round(float(res.loglike), 4)
    all_coefficients[f'airline_{nome}'] = coefs

output_path = os.path.join(OUTPUT_DIR, 'arima_coefficients.json')
with open(output_path, 'w') as f:
    json.dump(all_coefficients, f, indent=2)

print(f'Coeficientes salvos em: {output_path}')
print(f'Total de modelos: {len(all_coefficients)}')
for k in all_coefficients:
    print(f'  - {k}')

Coeficientes salvos em: /home/guhaase/projetos/chronobox/examples/arima/outputs/arima_coefficients.json
Total de modelos: 9
  - nile_ARIMA(1,1,0)
  - nile_ARIMA(0,1,1)
  - nile_ARIMA(1,1,1)
  - nile_ARIMA(2,1,1)
  - airline_ARIMA(1,1,0)
  - airline_ARIMA(0,1,1)
  - airline_ARIMA(1,1,1)
  - airline_ARIMA(2,1,1)
  - airline_ARIMA(2,1,2)


In [22]:
# Salvar previsoes em outputs/arima_forecasts.csv
forecasts_data = []

# Previsao do melhor modelo no Nile
fc_nile = resultados['ARIMA(1,1,1)'].forecast(steps=10, alpha=0.05)
for i in range(len(fc_nile['forecast'])):
    forecasts_data.append({
        'dataset': 'nile',
        'model': 'ARIMA(1,1,1)',
        'step': i + 1,
        'forecast': round(float(fc_nile['forecast'][i]), 4),
        'lower_95': round(float(fc_nile['lower'][i]), 4),
        'upper_95': round(float(fc_nile['upper'][i]), 4)
    })

# Previsao dos modelos no airline
for nome in ['ARIMA(1,1,1)', 'ARIMA(2,1,2)']:
    fc_air = resultados_air[nome].forecast(steps=12, alpha=0.05)
    for i in range(len(fc_air['forecast'])):
        forecasts_data.append({
            'dataset': 'airline',
            'model': nome,
            'step': i + 1,
            'forecast': round(float(fc_air['forecast'][i]), 6),
            'lower_95': round(float(fc_air['lower'][i]), 6),
            'upper_95': round(float(fc_air['upper'][i]), 6)
        })

df_forecasts = pd.DataFrame(forecasts_data)
output_path = os.path.join(OUTPUT_DIR, 'arima_forecasts.csv')
df_forecasts.to_csv(output_path, index=False)

print(f'Previsoes salvas em: {output_path}')
print(f'Total de linhas: {len(df_forecasts)}')
print(df_forecasts.head(10))

Previsoes salvas em: /home/guhaase/projetos/chronobox/examples/arima/outputs/arima_forecasts.csv
Total de linhas: 34
  dataset         model  step  forecast  lower_95  upper_95
0    nile  ARIMA(1,1,1)     1   76.1809 -199.3967  351.7584
1    nile  ARIMA(1,1,1)     2   19.3786 -304.8326  343.5899
2    nile  ARIMA(1,1,1)     3    4.9295 -322.1798  332.0387
3    nile  ARIMA(1,1,1)     4    1.2539 -326.0419  328.5498
4    nile  ARIMA(1,1,1)     5    0.3190 -326.9890  327.6269
5    nile  ARIMA(1,1,1)     6    0.0811 -327.2276  327.3898
6    nile  ARIMA(1,1,1)     7    0.0206 -327.2881  327.3294
7    nile  ARIMA(1,1,1)     8    0.0053 -327.3035  327.3140
8    nile  ARIMA(1,1,1)     9    0.0013 -327.3074  327.3101
9    nile  ARIMA(1,1,1)    10    0.0003 -327.3084  327.3091


## Conclusao

Neste notebook solution, resolvemos todos os exercicios:

1. **Exercicio 1**: Ajustamos ARIMA(1,1,1) e ARIMA(2,1,2) no airline.csv, confirmando que ARIMA puro nao captura sazonalidade adequadamente.
2. **Exercicio 2**: Demonstramos que sobre-diferenciacao (d=2) aumenta dramaticamente a variancia das previsoes.

**Outputs salvos:**
- `outputs/arima_coefficients.json`: coeficientes de todos os modelos
- `outputs/arima_forecasts.csv`: previsoes com intervalos de confianca